# 第2章 机器学习基础 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter2/Chapter2.md]]
- [[课程/深度学习/课堂笔记/2026-09-02-第2节-机器学习基础概念与模型构建教学.md]]
- [[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]
- [[课程/深度学习/演示代码/Chapter2.py]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## T2-A 有理函数回归：验证选次数后重拟合

**来源与性质**：[[课程/深度学习/PPT/Chapter2/Chapter2.md]]例1与验证集；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C2-01、C2-02。新编限时题，增加噪声、独立验证与最终重拟合，不冒充原例数据。


**100分，30分钟，一道完整编程题。** 从空白开始，只有下列接口约定，不提供程序框架。

1. **数据与划分（15分）**：`rng=np.random.default_rng(20261201)`，依次生成 `degree_a_x=rng.uniform(-1,1,55)` 和55个 `[-.025,.025)` 噪声；`degree_a_y=1/(x*x+x+1)+noise`。前18点训练、接12点验证、后25点测试，不能排序后再划分。
2. **选型接口（30分）**：实现 `select_degree(x_train,y_train,x_val,y_val,degrees)`，每个候选用升幂设计矩阵和最小二乘拟合；返回 `(chosen_degree,table,coefs)`。table每行 `[degree,train_rmse,val_rmse]`，保持degrees的顺序；coefs以次数作键。按验证RMSE最小选择，恰好同分选较小次数。实际候选为 `[1,2,4,8]`，保存 `degree_a_chosen,degree_a_table,degree_a_coefs`。接口不能接收测试标签。
3. **重拟合与评价（25分）**：用所选次数、全部前30点重拟合，保存升幂系数 `degree_a_final_coef`，后25点预测 `degree_a_test_pred`；保存并打印 `degree_a_train_rmse`（最终30点）、`degree_a_test_rmse`、`degree_a_gap_abs`（课程泛化差距）及 `degree_a_gap_signed`（额外有符号诊断）。所有RMSE对带噪观测标签计算。不得用测试结果改变次数。
4. **图表（15分）**：在[-1,1]的201点网格画真实有理函数、最终学习器及三组观测点；图例、轴标签清楚，创建本地输出目录后保存 `output/Q201A.png`。
5. **解释（15分）**：说明验证与测试的职责、为何最终训练误差按30点计算，以及“多项式对参数线性”并不等于图像是一条直线。

全部lstsq指定 `rcond=None`；系数／误差核验 `atol=1e-9`。图和解释人工评分。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
def select_degree(x_train,y_train,x_val,y_val,degrees):
    rows=[]; coefs={}
    for p in degrees:
        A=np.column_stack([x_train**j for j in range(p+1)])
        Av=np.column_stack([x_val**j for j in range(p+1)])
        c=np.linalg.lstsq(A,y_train,rcond=None)[0]; coefs[p]=c
        er=float(np.sqrt(np.mean((A@c-y_train)**2)))
        ev=float(np.sqrt(np.mean((Av@c-y_val)**2)))
        rows.append([p,er,ev])
    best=min(rows,key=lambda row:(row[2],row[0]))
    return int(best[0]),np.array(rows),coefs
_rng=np.random.default_rng(20261201)
degree_a_x=_rng.uniform(-1,1,55)
degree_a_y=1/(degree_a_x**2+degree_a_x+1)+_rng.uniform(-.025,.025,55)
degree_a_chosen,degree_a_table,degree_a_coefs=select_degree(degree_a_x[:18],degree_a_y[:18],degree_a_x[18:30],degree_a_y[18:30],[1,2,4,8])
_p=degree_a_chosen
_A=np.column_stack([degree_a_x[:30]**j for j in range(_p+1)])
_At=np.column_stack([degree_a_x[30:]**j for j in range(_p+1)])
degree_a_final_coef=np.linalg.lstsq(_A,degree_a_y[:30],rcond=None)[0]
degree_a_test_pred=_At@degree_a_final_coef
degree_a_train_rmse=float(np.sqrt(np.mean((_A@degree_a_final_coef-degree_a_y[:30])**2)))
degree_a_test_rmse=float(np.sqrt(np.mean((degree_a_test_pred-degree_a_y[30:])**2)))
degree_a_gap_signed=degree_a_test_rmse-degree_a_train_rmse
degree_a_gap_abs=abs(degree_a_gap_signed)
print('degree train18_RMSE validation12_RMSE',degree_a_table,sep='; ')
print('selected:',degree_a_chosen)
print('final_train30_RMSE test25_RMSE gap_abs gap_signed')
print(degree_a_train_rmse,degree_a_test_rmse,degree_a_gap_abs,degree_a_gap_signed)
Path('output').mkdir(exist_ok=True)
plt.figure(); _g=np.linspace(-1,1,201)
plt.plot(_g,1/(_g**2+_g+1),'k--',label='target')
plt.plot(_g,np.column_stack([_g**j for j in range(_p+1)])@degree_a_final_coef,label='selected refit')
for _a,_b,_name in [(0,18,'train'),(18,30,'validation'),(30,55,'test')]:
    plt.plot(degree_a_x[_a:_b],degree_a_y[_a:_b],'.',label=_name)
plt.xlabel('x'); plt.ylabel('observed y'); plt.legend()
plt.savefig('output/Q201A.png'); plt.close()
print('Validation selects the degree; testing is reserved for the final model.')
print('The final fit uses 30 samples, so its training RMSE must use those same 30 labels.')
print('The model is linear in its coefficients, although it can be nonlinear in x.')


### 解析与点评

本题的候选参数来自18点，最终参数来自30点，二者不能混用。测试标签虽然程序中已生成，也不能参与choose逻辑；公开自检核对代表性数值，完整职责边界还需人工阅读。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
_x=np.array([-1.,-.5,0.,.5,1.]); _y=1+2*_x+3*_x**2
_v=np.array([-.75,.25,.75]); _yv=1+2*_v+3*_v**2
_p,_t,_c=select_degree(_x,_y,_v,_yv,[0,1,2])
assert _p==2 and np.allclose(_c[2],[1,2,3],atol=1e-9) and np.asarray(_t)[2,2]<1e-10
_rng=np.random.default_rng(20261201); _x=_rng.uniform(-1,1,55)
_y=1/(_x**2+_x+1)+_rng.uniform(-.025,.025,55)
assert np.allclose(degree_a_x,_x) and np.allclose(degree_a_y,_y)
_check_degree_a_table = np.asarray(degree_a_table, dtype=float)
assert np.array_equal(_check_degree_a_table[:,0],[1,2,4,8])
_best=min(_check_degree_a_table.tolist(),key=lambda r:(r[2],r[0]))
assert degree_a_chosen==int(_best[0])
for _i,_p in enumerate([1,2,4,8]):
    _A=np.column_stack([_x[:18]**j for j in range(_p+1)])
    _Av=np.column_stack([_x[18:30]**j for j in range(_p+1)])
    _c=degree_a_coefs[_p]
    assert np.linalg.norm(_A.T@(_A@_c-_y[:18]))<1e-9
    assert np.isclose(_check_degree_a_table[_i,2],np.sqrt(np.mean((_Av@_c-_y[18:30])**2)))
_p=degree_a_chosen; _A=np.column_stack([_x[:30]**j for j in range(_p+1)])
_At=np.column_stack([_x[30:]**j for j in range(_p+1)])
assert np.linalg.norm(_A.T@(_A@degree_a_final_coef-_y[:30]))<1e-9
assert np.allclose(degree_a_test_pred,_At@degree_a_final_coef,atol=1e-9)
assert np.isclose(degree_a_train_rmse,np.sqrt(np.mean((_A@degree_a_final_coef-_y[:30])**2)))
assert np.isclose(degree_a_test_rmse,np.sqrt(np.mean((degree_a_test_pred-_y[30:])**2)))
assert np.isclose(degree_a_gap_abs,abs(degree_a_test_rmse-degree_a_train_rmse))
assert np.isclose(degree_a_gap_signed,degree_a_test_rmse-degree_a_train_rmse)
assert Path('output/Q201A.png').stat().st_size>1000

print("自检通过")
